In [11]:
"""
Final: Travel & Tourism Recommendation System (Jupyter Notebook / Google Colab)
"""

import os
import math
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# -----------------------
# USER PARAMETERS - UPDATED FOR JUPYTER/COLAB
# -----------------------
# Upload your CSV file to Colab or put it in the same directory as your notebook
DATA_PATH = "C:/Users/gamne/Downloads/travel_tourism_refined_dataset.csv"
RANDOM_STATE = 42

# Create output directories
os.makedirs("plots", exist_ok=True)
os.makedirs("models", exist_ok=True)

# -----------------------
# Helpers: saving / plotting
# -----------------------
def save_fig(fig, name):
    path = os.path.join("plots", name)
    fig.savefig(path, bbox_inches="tight", dpi=140)
    plt.close(fig)
    print(f"Saved plot: {path}")

def quick_info(df):
    print("Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    print("\nDtypes:\n", df.dtypes)
    print("\nMissing (top 20):\n", df.isna().sum().sort_values(ascending=False).head(20))
    print("\nSample rows:\n", df.head(5))

# -----------------------
# Parse age_group_distribution into numeric columns
# Example cell: "2 kids, 2 adults, 2 seniors"
# -----------------------
def parse_age_group_column(df, col="age_group_distribution"):
    if col not in df.columns:
        return df
    def parse_row(val):
        kids = adults = seniors = 0
        if pd.isna(val):
            return pd.Series([kids, adults, seniors])
        s = str(val)
        parts = [p.strip().lower() for p in s.split(",")]
        for p in parts:
            if "kid" in p:
                try:
                    kids = int(p.split()[0])
                except:
                    # fallback: extract digits
                    import re
                    m = re.search(r'\d+', p)
                    kids = int(m.group()) if m else 0
            elif "adult" in p:
                try:
                    adults = int(p.split()[0])
                except:
                    import re
                    m = re.search(r'\d+', p)
                    adults = int(m.group()) if m else 0
            elif "senior" in p or "elder" in p:
                try:
                    seniors = int(p.split()[0])
                except:
                    import re
                    m = re.search(r'\d+', p)
                    seniors = int(m.group()) if m else 0
        return pd.Series([kids, adults, seniors])
    new_cols = df[col].apply(parse_row)
    new_cols.columns = ["kids_count", "adults_count", "seniors_count"]
    df = pd.concat([df.drop(columns=[col]), new_cols], axis=1)
    return df

# -----------------------
# Detect attractions column (heuristic)
# -----------------------
def detect_attractions_column(df):
    # try a few likely names
    candidates = [c for c in df.columns if any(k in c.lower() for k in ["attract", "place", "popular", "top", "sights", "places"])]
    return candidates[0] if candidates else None

# -----------------------
# Build suggestions mapping from dataset
# -----------------------
def build_suggestions_from_dataset(df, dest_col="destination_type", city_col_candidates=None, attr_col=None):
    # city_col_candidates: list of possible city columns (heuristic)
    if dest_col not in df.columns:
        return {}
    # heuristics for city column
    if city_col_candidates is None:
        city_candidates = [c for c in df.columns if any(k in c.lower() for k in ["city", "destination_city", "place", "location"])]
    else:
        city_candidates = city_col_candidates
    city_col = city_candidates[0] if city_candidates else None

    suggestions = {}
    for dtype, grp in df.groupby(dest_col):
        cities = []
        attractions = {}
        if city_col:
            cities = grp[city_col].dropna().unique().tolist()
        # try fill attractions from attr_col or from a column that looks like strings of places
        if attr_col and attr_col in grp.columns:
            # attr cells might be comma-separated lists; collect by city if possible
            for _, row in grp[[city_col, attr_col]].dropna().iterrows():
                c = row.get(city_col, "Unknown")
                raw = str(row[attr_col])
                places = [p.strip() for p in raw.split(",") if p.strip()]
                if c not in attractions:
                    attractions[c] = places
                else:
                    attractions[c].extend(places)
            # deduplicate lists
            attractions = {k: sorted(list(set(v))) for k, v in attractions.items()}
        suggestions[dtype] = {"cities": cities, "attractions": attractions}
    return suggestions

# -----------------------
# EDA plots (simple)
# -----------------------
def do_eda_and_save(df):
    # numeric histograms
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in num_cols:
        fig = plt.figure(figsize=(6,4))
        plt.hist(df[col].dropna(), bins=25)
        plt.title(f"Histogram: {col}")
        plt.xlabel(col)
        save_fig(fig, f"hist_{col}.png")
    # correlation heatmap (simple)
    if len(num_cols) > 1:
        corr = df[num_cols].corr()
        fig = plt.figure(figsize=(10,8))
        plt.imshow(corr, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
        plt.colorbar()
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
        plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
        plt.title("Correlation (numeric)")
        save_fig(fig, "correlation_matrix.png")

# -----------------------
# Build preprocessing ColumnTransformer and return it plus feature lists
# -----------------------
def prepare_preprocessor(df, features):
    numeric_features = df[features].select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = df[features].select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers = [
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_features, categorical_features

# -----------------------
# Train and compare regression models
# -----------------------
def train_regression_models(df, preprocessor, features, target_col):
    X = df[features].copy()
    y = df[target_col].copy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

    models = {
        "Linear": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(max_iter=5000),
        "RandomForest": RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE),
        "SVR": SVR()
    }
    results = {}
    for name, m in models.items():
        pipe = Pipeline([("pre", preprocessor), ("model", m)])
        try:
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)
            rmse = math.sqrt(mean_squared_error(y_test, preds))
            r2 = r2_score(y_test, preds)
            results[name] = {"model": pipe, "RMSE": rmse, "R2": r2}
            # save model
            model_path = os.path.join("models", f"reg_{name}.pkl")
            pickle.dump(pipe, open(model_path, "wb"))
            print(f"[Reg] {name}: RMSE={rmse:.2f}, R2={r2:.3f}")
        except Exception as e:
            print(f"[Reg] {name} failed: {e}")
    # save comparison plot
    if results:
        comp = pd.DataFrame({k: {"RMSE": v["RMSE"], "R2": v["R2"]} for k, v in results.items()}).T
        # R2 bar
        fig = plt.figure(figsize=(8,4))
        comp["R2"].plot(kind="bar")
        plt.title("Regression models R2")
        save_fig(fig, "regression_r2.png")
        fig = plt.figure(figsize=(8,4))
        comp["RMSE"].plot(kind="bar")
        plt.title("Regression models RMSE")
        save_fig(fig, "regression_rmse.png")
    return results

# -----------------------
# Train and compare classification models
# -----------------------
def train_classification_models(df, preprocessor, features, target_col):
    X = df[features].copy()
    y = df[target_col].copy().astype(str)
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=RANDOM_STATE)

    models = {
        "Logistic": LogisticRegression(max_iter=2000),
        "RandomForest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE),
        "SVM": SVC(probability=True),
        "KNN": KNeighborsClassifier(n_neighbors=7)
    }
    results = {}
    for name, m in models.items():
        pipe = Pipeline([("pre", preprocessor), ("model", m)])
        try:
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average="weighted")
            results[name] = {"model": pipe, "Accuracy": acc, "F1": f1}
            model_path = os.path.join("models", f"cls_{name}.pkl")
            pickle.dump(pipe, open(model_path, "wb"))
            print(f"[Cls] {name}: Acc={acc:.3f}, F1={f1:.3f}")
        except Exception as e:
            print(f"[Cls] {name} failed: {e}")
    # comparison plot
    if results:
        comp = pd.DataFrame({k: {"Accuracy": v["Accuracy"], "F1": v["F1"]} for k, v in results.items()}).T
        fig = plt.figure(figsize=(8,4))
        comp["Accuracy"].plot(kind="bar")
        plt.title("Classification Accuracy")
        save_fig(fig, "classification_accuracy.png")
    return results, le

# -----------------------
# Clustering (numeric features only)
# -----------------------
def run_clustering(df, numeric_features, k_min=2, k_max=8):
    X = df[numeric_features].fillna(df[numeric_features].median())
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    sils = {}
    for k in range(k_min, min(k_max, len(X)) + 1):
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE)
        labels = kmeans.fit_predict(Xs)
        try:
            sil = silhouette_score(Xs, labels)
        except:
            sil = -1
        sils[k] = sil
    # pick best
    best_k = max(sils, key=sils.get)
    final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE).fit(Xs)
    # save elbow/silhouette visual
    fig = plt.figure(figsize=(8,4))
    plt.plot(list(sils.keys()), list(sils.values()), marker="o")
    plt.title("Silhouette scores by k")
    plt.xlabel("k")
    plt.ylabel("silhouette")
    save_fig(fig, "silhouette_by_k.png")

    # PCA 2D plot
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    reduced = pca.fit_transform(Xs)
    fig = plt.figure(figsize=(8,6))
    scatter = plt.scatter(reduced[:,0], reduced[:,1], c=final_kmeans.labels_, cmap="tab10", alpha=0.6)
    plt.title(f"KMeans clusters (k={best_k}) PCA projection")
    save_fig(fig, "clusters_pca.png")

    return final_kmeans, scaler, numeric_features

# -----------------------
# Recommendation function
# -----------------------
def recommend_trip_from_models(user_input: dict, reg_model_pipe, cls_model_pipe, cluster_model, cluster_scaler, num_features, label_encoder, suggestions_map):
    # user_input must contain same raw feature names used to train (not preprocessed)
    df_input = pd.DataFrame([user_input])

    # Predict budget (reg pipeline already includes preprocessor)
    try:
        budget_pred = reg_model_pipe.predict(df_input)[0]
    except Exception as e:
        raise RuntimeError(f"Regression prediction failed: {e}")

    # Predict destination type (cls pipeline includes preprocessor)
    try:
        cls_raw = cls_model_pipe.predict(df_input)[0]
        dest_type = label_encoder.inverse_transform([cls_raw])[0]
    except Exception as e:
        raise RuntimeError(f"Classification prediction failed: {e}")

    # Cluster: scale numeric features with cluster_scaler and predict
    df_num = df_input[num_features].fillna(0)
    Xs = cluster_scaler.transform(df_num)
    cluster_label = int(cluster_model.predict(Xs)[0])

    # Gather suggestions
    suggested_cities = []
    nearby_attractions = {}
    if suggestions_map and dest_type in suggestions_map:
        suggested_cities = suggestions_map[dest_type].get("cities", [])[:10]
        nearby_attractions = suggestions_map[dest_type].get("attractions", {})

    return {
        "Predicted_Budget": round(float(budget_pred), 2),
        "Predicted_Destination_Type": dest_type,
        "Suggested_Cities": suggested_cities,
        "Nearby_Attractions": nearby_attractions,
        "Cluster_Group": cluster_label
    }

# -----------------------
# Main: orchestrate everything
# -----------------------
def main():
    # load
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}\n\nPlease upload your CSV file or update DATA_PATH variable.")
    df = pd.read_csv(DATA_PATH)
    print("Dataset Loaded:", df.shape)
    quick_info(df)

    # detect regression and classification targets
    # From earlier inspection your dataset uses 'total_budget' as budget
    if "total_budget" in df.columns:
        budget_col = "total_budget"
    elif "budget" in df.columns:
        budget_col = "budget"
    else:
        # guess last numeric column
        numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols_all:
            raise RuntimeError("No numeric columns found for regression target.")
        budget_col = numeric_cols_all[-1]
        print("Using fallback budget column:", budget_col)

    if "destination_type" in df.columns:
        dest_col = "destination_type"
    else:
        # fallback - pick most plausible categorical
        cats = df.select_dtypes(include=['object','category']).columns.tolist()
        if not cats:
            raise RuntimeError("No categorical columns found for destination_type.")
        dest_col = cats[0]
        print("Using fallback destination column:", dest_col)

    # 1) Parse age_group_distribution if present
    if "age_group_distribution" in df.columns:
        df = parse_age_group_column(df, col="age_group_distribution")
        print("Parsed age_group_distribution -> kids_count/adults_count/seniors_count")

    # 2) Detect attractions column & possible city column
    attractions_col = detect_attractions_column(df)
    city_candidates = [c for c in df.columns if "city" in c.lower() or "destination_city" in c.lower() or "location" in c.lower()]
    city_col = city_candidates[0] if city_candidates else None
    print("Attractions column detected:", attractions_col)
    print("City column detected:", city_col)

    # 3) Build suggestions map
    suggestions_map = build_suggestions_from_dataset(df, dest_col, city_col_candidates=[city_col] if city_col else None, attr_col=attractions_col)

    # 4) Choose features (exclude targets and text attraction column)
    excludes = {budget_col, dest_col, attractions_col, "id", "recommended"}
    features = [c for c in df.columns if c not in excludes and c is not None]
    print("Features used for modelling (first 40):", features[:40])

    # 5) Preprocessor
    preprocessor, numeric_features, categorical_features = prepare_preprocessor(df, features)
    print("Numeric features:", numeric_features)
    print("Categorical features:", categorical_features)

    # 6) EDA
    do_eda_and_save(df)

    # 7) Train regression models
    reg_results = train_regression_models(df, preprocessor, features, budget_col)
    # pick best reg (lowest RMSE)
    best_reg_name = min(reg_results.keys(), key=lambda k: reg_results[k]["RMSE"])
    best_reg_pipe = reg_results[best_reg_name]["model"]
    print("Best regression model:", best_reg_name)

    # 8) Train classification models
    cls_results, label_enc = None, None
    cls_results, cls_comp = None, None
    cls_models, label_enc = train_classification_models(df, preprocessor, features, dest_col)
    if cls_models:
        best_cls_name = max(cls_models.keys(), key=lambda k: cls_models[k]["Accuracy"])
        best_cls_pipe = cls_models[best_cls_name]["model"]
        print("Best classification model:", best_cls_name)
    else:
        raise RuntimeError("No classification models trained successfully.")

    # 9) Clustering (numeric features only)
    if len(numeric_features) < 1:
        raise RuntimeError("No numeric features available for clustering.")
    cluster_model, cluster_scaler, cluster_num_features = run_clustering(df, numeric_features)
    # save clustering model
    model_path = os.path.join("models", "kmeans_cluster.pkl")
    pickle.dump(cluster_model, open(model_path, "wb"))


    # 10) Final recommendation example (use a user profile - adapt as needed)
    # Build a "safe" user_data dict: include all features with defaults if missing
    sample_user = {}
    for f in features:
        # if feature in dataframe sample take median/mode as default else 0
        if f in df.columns:
            if pd.api.types.is_numeric_dtype(df[f]):
                sample_user[f] = float(df[f].median(skipna=True)) if df[f].notna().any() else 0.0
            else:
                sample_user[f] = df[f].mode().iloc[0] if df[f].notna().any() else ""
        else:
            sample_user[f] = 0

    # Overwrite some user values to demo (you can modify these)
    # Only set keys present in features
    demo_inputs = {
        "family_size": 2,
        "income_level": 120000,           # if your dataset uses 'income_level' numeric
        "average_stay_days": 4,
        "preferred_travel_month": "June",
        "kids_count": 0,
        "adults_count": 2,
        "seniors_count": 0,
        "activity_preference": "",  # example categorical
        # set other preference flags if available in your features:
    }
    for k,v in demo_inputs.items():
        if k in sample_user:
            sample_user[k] = v

    # Run recommendation
    result = recommend_trip_from_models(sample_user, best_reg_pipe, best_cls_pipe, cluster_model, cluster_scaler, cluster_num_features, label_enc, suggestions_map)

    # Print as requested
    print("\n=== Final Recommendation for User ===")
    print(f"Predicted_Budget: {result['Predicted_Budget']}")
    print(f"Predicted_Destination_Type: {result['Predicted_Destination_Type']}")
    print(f"Suggested_Cities: {result['Suggested_Cities']}")
    print("Nearby_Attractions: {")
    for city, places in result["Nearby_Attractions"].items():
        print(f"   '{city}': {places}")
    print("}")
    print(f"Cluster_Group: {result['Cluster_Group']}")

    # Save final recommendation as JSON
    try:
        out_path = "final_recommendation.json"
        pd.Series(result).to_json(out_path)
        print("Saved final recommendation to", out_path)
    except Exception as e:
        print("Failed saving recommendation:", e)


if __name__ == "__main__":
    main()

Dataset Loaded: (200, 25)
Shape: (200, 25)

Columns: ['family_size', 'age_group_distribution', 'income_level', 'preferred_travel_month', 'home_state', 'travel_mode_preference', 'average_stay_days', 'accommodation_preference', 'food_preference', 'activity_preference', 'destination_city', 'destination_type', 'state_of_destination', 'distance_from_home', 'average_weather', 'crowd_level', 'safety_index', 'nearest_airport_km', 'avg_hotel_cost_per_night', 'popular_attractions', 'transport_cost', 'stay_cost', 'food_cost', 'activity_cost', 'total_budget']

Dtypes:
 family_size                  int64
age_group_distribution      object
income_level                object
preferred_travel_month      object
home_state                  object
travel_mode_preference      object
average_stay_days            int64
accommodation_preference    object
food_preference             object
activity_preference         object
destination_city            object
destination_type            object
state_of_destina